In [1]:
#csp.py

from typing import List, Set, Optional 

import copy

#---------- Groupes ----------

class Groupe:
    def __init__(self):
        self.cellules: List["Cellule"] = []

    def chiffres_utilises(self) -> Set[int]:
        return {c.valeur for c in self.cellules if c.valeur != 0}

    def chiffres_manquants(self) -> Set[int]:
        return set(range(1, 10)) - self.chiffres_utilises()

    def complet(self) -> bool:
        return len(self.chiffres_manquants()) == 0

class Ligne(Groupe):
    pass

class Colonne(Groupe):
    pass

class Bloc(Groupe):
    pass

#---------- Cellule ----------

class Cellule:
    def __init__(self, ligne: int, colonne: int, valeur: int = 0, fixe: bool = False):
        self.ligne = ligne
        self.colonne = colonne
        self.bloc = (ligne // 3) * 3 + (colonne // 3)
        self.valeur = valeur
        self.fixe = fixe
        self.disponibles: Set[int] = set()
        self.nb_candidats = 0
        self.grp_ligne: Optional[Ligne] = None
        self.grp_colonne: Optional[Colonne] = None
        self.grp_bloc: Optional[Bloc] = None

    def est_vide(self) -> bool:
        return self.valeur == 0

    def maj_disponibles(self):
        if not self.est_vide():
            self.disponibles = set()
            self.nb_candidats = 0
        else:
            possibles = set(range(1, 10))
            possibles &= self.grp_ligne.chiffres_manquants()
            possibles &= self.grp_colonne.chiffres_manquants()
            possibles &= self.grp_bloc.chiffres_manquants()
            self.disponibles = possibles
            self.nb_candidats = len(possibles)

    def voisins(self):
        v = {c for c in self.grp_ligne.cellules if c.est_vide() and c != self}
        v |= {c for c in self.grp_colonne.cellules if c.est_vide() and c != self}
        v |= {c for c in self.grp_bloc.cellules if c.est_vide() and c != self}
        return v

#---------- Sudoku CSP ----------

class SudokuCSP:
    def __init__(self, grille_initiale: List[List[int]], mode="MRV"):
        self.mode = mode   # "MRV", "MRV+Degree", "MRV+Degree+LCV", "ForwardChecking"
        self.backtracks = 0
        self.lignes = [Ligne() for _ in range(9)]
        self.colonnes = [Colonne() for _ in range(9)]
        self.blocs = [Bloc() for _ in range(9)]

        self.grille: List[List[Cellule]] = []
        for i in range(9):
            row = []
            for j in range(9):
                val = grille_initiale[i][j]
                cell = Cellule(i, j, val, fixe=(val != 0))
                cell.grp_ligne = self.lignes[i]
                cell.grp_colonne = self.colonnes[j]
                cell.grp_bloc = self.blocs[cell.bloc]
                self.lignes[i].cellules.append(cell)
                self.colonnes[j].cellules.append(cell)
                self.blocs[cell.bloc].cellules.append(cell)
                row.append(cell)
            self.grille.append(row)

    # ---- Candidats ----

    def _candidates_vides(self) -> List[Cellule]:
        cells = [c for row in self.grille for c in row if c.est_vide()]
        for c in cells:
            c.maj_disponibles()
        return cells

    # ---- Heuristiques ----

    def _choisir_cellule(self) -> Optional[Cellule]:
        candidates = self._candidates_vides()
        if not candidates:
            return None

        zero = [c for c in candidates if c.nb_candidats == 0]
        if zero:
            return zero[0]

        # MRV : cellule avec le moins de candidats
        min_len = min(c.nb_candidats for c in candidates)
        mrv = [c for c in candidates if c.nb_candidats == min_len]

        if self.mode == "MRV":
            return mrv[0]

        # Degree heuristic : cellule qui contraint le plus de voisins
        if self.mode in ("MRV+Degree", "MRV+Degree+LCV", "ForwardChecking"):
            def deg(cell: Cellule): return len(cell.voisins())
            best = max(mrv, key=deg)
            return best

    # ---- Ordre de valeurs (LCV) ----

    def _order_values(self, cell: Cellule) -> List[int]:
        if self.mode == "MRV+Degree+LCV":
            # LCV : valeur qui élimine le moins de candidats chez les voisins
            def impact(val):
                score = 0
                for v in cell.voisins():
                    if val in v.disponibles:
                        score += 1
                return score
            return sorted(cell.disponibles, key=impact)
        else:
            return sorted(cell.disponibles)

    # ---- Solveur ----

    def solve(self) -> bool:
        cell = self._choisir_cellule()
        if cell is None:
            return True
        if cell.nb_candidats == 0 and cell.est_vide():
            self.backtracks += 1
            return False

        for val in self._order_values(cell):
            cell.valeur = val

            if self.mode == "ForwardChecking":
                # Vérifie qu'aucun voisin ne devient sans candidat
                consistent = True
                for v in cell.voisins():
                    v.maj_disponibles()
                    if v.nb_candidats == 0:
                        consistent = False
                        break
                if not consistent:
                    cell.valeur = 0
                    self.backtracks += 1
                    continue

            if self.solve():
                return True

            cell.valeur = 0
            self.backtracks += 1
        return False

    # ---- Outils ----

    def solution(self) -> List[List[int]]:
        return [[c.valeur for c in row] for row in self.grille]

    def afficher(self):
        for i in range(9):
            if i % 3 == 0 and i != 0:
                print("------+-------+------")
            line = []
            for j in range(9):
                if j % 3 == 0 and j != 0:
                    line.append("|")
                v = self.grille[i][j].valeur
                line.append(str(v) if v != 0 else ".")
            print(" ".join(line))
        print()




In [2]:
#from csp import SudokuCSP

grille = [
    [3,9,6,0,0,0,0,1,2],
    [7,4,0,0,0,0,8,0,0],
    [0,0,1,0,9,5,0,0,0],
    [0,0,0,0,0,0,0,0,0],
    [0,0,0,5,3,8,0,0,0],
    [0,0,0,0,0,0,0,0,0],
    [0,0,0,3,6,0,7,0,0],
    [0,0,0,0,0,0,0,0,0],
    [4,8,0,0,0,0,0,0,0],
]

grille_hard2 = [
    [0,0,0, 0,0,6, 0,3,1],
    [2,5,7, 0,0,0, 0,0,0],
    [0,0,0, 0,0,0, 0,0,0],
    [7,4,0, 6,9,3, 2,0,0],
    [0,0,0, 0,0,1, 0,9,0],
    [0,0,6, 4,0,0, 0,5,0],
    [3,7,0, 0,1,4, 9,6,0],
    [4,2,9, 0,6,7, 0,0,0],
    [0,0,1, 0,0,9, 0,0,0],	]
    
grid = grille_hard2  #diabolique

for mode in ("MRV", "MRV+Degree", "MRV+Degree+LCV", "ForwardChecking"):
    s = SudokuCSP(grid, mode=mode)
    print(mode)
    s.solve()
    s.afficher()
    print("Backtracks:", s.backtracks)

MRV
9 8 4 | 2 7 6 | 5 3 1
2 5 7 | 1 3 8 | 6 4 9
6 1 3 | 9 4 5 | 8 2 7
------+-------+------
7 4 5 | 6 9 3 | 2 1 8
8 3 2 | 7 5 1 | 4 9 6
1 9 6 | 4 8 2 | 7 5 3
------+-------+------
3 7 8 | 5 1 4 | 9 6 2
4 2 9 | 3 6 7 | 1 8 5
5 6 1 | 8 2 9 | 3 7 4

Backtracks: 0
MRV+Degree
9 8 4 | 2 7 6 | 5 3 1
2 5 7 | 1 3 8 | 6 4 9
6 1 3 | 9 4 5 | 8 2 7
------+-------+------
7 4 5 | 6 9 3 | 2 1 8
8 3 2 | 7 5 1 | 4 9 6
1 9 6 | 4 8 2 | 7 5 3
------+-------+------
3 7 8 | 5 1 4 | 9 6 2
4 2 9 | 3 6 7 | 1 8 5
5 6 1 | 8 2 9 | 3 7 4

Backtracks: 11
MRV+Degree+LCV
9 8 4 | 2 7 6 | 5 3 1
2 5 7 | 1 3 8 | 6 4 9
6 1 3 | 9 4 5 | 8 2 7
------+-------+------
7 4 5 | 6 9 3 | 2 1 8
8 3 2 | 7 5 1 | 4 9 6
1 9 6 | 4 8 2 | 7 5 3
------+-------+------
3 7 8 | 5 1 4 | 9 6 2
4 2 9 | 3 6 7 | 1 8 5
5 6 1 | 8 2 9 | 3 7 4

Backtracks: 11
ForwardChecking
9 8 4 | 2 7 6 | 5 3 1
2 5 7 | 1 3 8 | 6 4 9
6 1 3 | 9 4 5 | 8 2 7
------+-------+------
7 4 5 | 6 9 3 | 2 1 8
8 3 2 | 7 5 1 | 4 9 6
1 9 6 | 4 8 2 | 7 5 3
------+-------+------
3 7 8